# SemantyFish Cleanup

## Imports, file loading, inspection

In [162]:
import json, pprint, re
import pandas as pd

In [16]:
with open("all_species_data.json", encoding="utf-8") as f:
    df = json.load(f)

pprint.pprint(df[0])

{'air_breathing_status': 'WaterAssumed',
 'aquarium_demand': {'details': None, 'value': 'never/rarely'},
 'body_shape': 'fusiform / normal',
 'brackish_water_environment': True,
 'catching_methods': {'main_method': 'seines',
                      'main_method_using_fao_name': None,
                      'other_methods': ['gillnets',
                                        'other method',
                                        'traps',
                                        'seines',
                                        'trawls']},
 'comments': 'Adults occur in a wide variety of freshwater habitats like '
             'rivers, lakes, sewage canals and irrigation channels (Ref. '
             '28714). They do not do well in pure salt water, but  able to '
             'survive in brackish water (Ref. 52307). Mainly diurnal. They '
             'feed mainly on phytoplankton or benthic algae. Additionally, '
             'insect larvae are of some importance, as are aufwuchs and '
   

In [45]:
for dim in df[0]['dimensions']:
    print(dim)

{'type': 'most shallow waters', 'value': 0.0, 'unit': 'meters', 'measurement_type': None, 'sex': None}
{'type': 'max length', 'value': 60.0, 'unit': 'centimeters', 'measurement_type': 'Standard Length', 'sex': 'male or unsexed'}
{'type': 'longevity wild', 'value': 9.0, 'unit': 'years', 'measurement_type': None, 'sex': None}
{'type': 'most deep waters', 'value': 20.0, 'unit': 'meters', 'measurement_type': None, 'sex': None}
{'type': 'max weight', 'value': 4324.0, 'unit': 'kilograms', 'measurement_type': None, 'sex': 'male or unsexed'}
{'type': 'common deep waters', 'value': 20.0, 'unit': 'meters', 'measurement_type': None, 'sex': None}
{'type': 'longevity captive', 'value': 12.0, 'unit': 'years', 'measurement_type': None, 'sex': None}


## Creating new df based on schema


In [126]:
test_df = pd.DataFrame(columns=['pet_id', 'pet_scientific_name','pet_vernacular_name','pet_genus', 'pet_family',
                        'pet_body_shape', 'pet_traits', 'pet_max_length', 'pet_max_weight', 'pet_longevity', 'pet_temperature',
                        'pet_migration_type', 'pet_danger', 'pet_is_native', 'pet_comments'])

In [167]:
test_df.dtypes

pet_id                  object
pet_scientific_name     object
pet_vernacular_name     object
pet_genus               object
pet_family              object
pet_body_shape          object
pet_traits              object
pet_max_length         float64
pet_max_weight         float64
pet_longevity          float64
pet_temperature         object
pet_migration_type      object
pet_danger              object
pet_is_native           object
pet_comments            object
dtype: object

## Adding pet ids

In [127]:
# Add pet_ids to test_df based on 'id' field in df
test_df['pet_id'] = [item['id'] for item in df]

# Id might also be used across other db, appending 'f' to indicate fishbase id
test_df['pet_id'] = test_df['pet_id'].astype(str) + 'f'

## Adding names

In [128]:
# Adding scientific name
test_df['pet_scientific_name'] = [item['scientific_name'] for item in df]

# Adding vernacular name, capitalising first letter of each word
# Leave missing common names as NaN
test_df['pet_vernacular_name'] = [item.get('common_name', None).title() if item.get('common_name') else None for item in df]  

## Adding family/genus

In [129]:
# Add genus and family based on 'genus' and 'family' fields in df, getting just genus_name/family_name from the inner dict
test_df['pet_genus'] = [item['genus']['genus_name'].title() for item in df]
test_df['pet_family'] = [item['family']['family_name'].title() for item in df]

## Adding body shape

In [130]:
# Add body shape
test_df['pet_body_shape'] = [item['body_shape'].title() for item in df]

## Adding measurements (length, weight, longevity)

In [131]:
# from the 'dimensions' field, extract the unique units for the inner dicts in the list where the 'type' is 'max length', 'max weight', 'longevity wild', or 'longevity captive'
units = set()
for item in df:     
    for dim in item['dimensions']:
        if dim['type'] in ['max length', 'max weight', 'longevity wild', 'longevity captive']:
            units.add(dim['unit'])
print(units)

{'kilograms', 'years', 'centimeters'}


In [132]:
# Add max length (cm), weight (kg), longevity (years)
# for max length/weight, retrieve the inner dict from 'dimensions' where 'type' = 'max_length', then get 'value' from that dict
test_df['pet_max_length'] = [next((dim['value'] for dim in item['dimensions'] if dim['type'] == 'max length'), None) for item in df]
test_df['pet_max_weight'] = [next((dim['value']/1000 for dim in item ['dimensions'] if dim['type'] == 'max weight'), None) for item in df]

# for longevity, retrieve the inner dict from 'dimensions' where 'type' = 'longevity_wild' or 'longevity_captive', 
# preferring the greater of the two values
def get_longevity(dimensions):
    longevity_wild = next((dim['value'] for dim in dimensions if dim['type'] == 'longevity wild'), None)
    longevity_captive = next((dim['value'] for dim in dimensions if dim['type'] == 'longevity captive'), None)
    if longevity_wild is not None and longevity_captive is not None:
        return max(longevity_wild, longevity_captive)
    elif longevity_captive is not None:
        return longevity_captive
    elif longevity_wild is not None:
        return longevity_wild
    else:
        return None
test_df['pet_longevity'] = [get_longevity(item['dimensions']) for item in df]



## Adding migration type

In [138]:
# Add migration type
test_df['pet_migration_type'] = [item['migration_type'].title() if item['migration_type'] else None for item in df]

## Adding pet danger

In [133]:
danger = set()
for item in df:
    danger.add(item['dangerous_species_indicator'])
print(danger)

{'poisonous to eat', 'reports of ciguatera poisoning', None, 'potential pest', 'traumatogenic', 'venomous', 'other', 'harmless'}


In [134]:
electric = set()
for item in df:
    electric.add(item['electrogenic'])
print(electric)

{'no special ability', 'electrosensing only', None, 'Weakly discharging', 'Electrosensing only', 'weakly discharging', 'strongly discharging'}


In [135]:
# pet_danger -->
# harmless: harmless, potential pest
# poisonous: poisonous to eat, reports of ciguatera poisoning
# venomous: venomous
# aggressive: traumatogenic
# unknown: null, None, other
def classify_danger(item):
    dangerStr = ''
    # Evaluate dangerous_species_indicator first
    if item['dangerous_species_indicator'] is None:
        dangerStr = 'Unknown'
    elif item['dangerous_species_indicator'].lower() in ['harmless', 'potential pest']:
        dangerStr = 'Harmless'
    elif item['dangerous_species_indicator'].lower() in ['poisonous to eat', 'reports of ciguatera poisoning']:
        dangerStr = 'Poisonous'
    elif item['dangerous_species_indicator'].lower() == 'venomous':
        dangerStr = 'Venomous'
    elif item['dangerous_species_indicator'].lower() == 'traumatogenic':
        dangerStr = 'Aggressive'


    # Evaluate electrogenic
    if item['electrogenic'] is None:
        dangerStr += '; Unknown Electrogenic'
    else:
        dangerStr += f"; {item['electrogenic'].title()}"

    return dangerStr

test_df['pet_danger'] = [classify_danger(item) for item in df]

In [136]:
# Setting native/non-native
test_df['pet_is_native'] = 'Not Native'

# Compare fish.csv against Species in fish.csv; if the fish is found in fish.csv AND the Occurrence == 'native', set pet_is_native to 'Native'
native_fish_df = pd.read_csv('fish.csv')
native_fish_df = native_fish_df[native_fish_df['Occurrence'].isin(['native', 'endemic'])]

test_df['pet_is_native'] = test_df.apply(lambda row: 'Native' if ((native_fish_df['Species'].str.lower() == row['pet_scientific_name'].lower()).any()) else 'Not Native', axis=1)

In [137]:
# Output to csv the fish that are in native_fish_df but not in test_df
native_fish_not_in_test_df = native_fish_df[~native_fish_df['Species'].isin(test_df['pet_scientific_name'])]
native_fish_not_in_test_df.to_csv('native_fish_not_in_test_df.csv', index=False)

## Extracting temperature from comments


In [159]:
import re
def get_temp(comment):
    if comment is None:
        return None
    
    matches = re.findall(r'(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)\s*°([CFcf])', comment)

    if not matches:
        return None

    ranges = []
    for low, high, unit in matches:
        
        width = float(high) - float(low)
        ranges.append((low, high, unit.upper(), width))

    # Pick range with smallest lower bound
    smallest = min(ranges, key=lambda x: x[3])  # x[3] = width of the range

    #print(f"{smallest[0]}-{smallest[1]} °{smallest[2]}")
    return(f"{smallest[0]}-{smallest[1]} °{smallest[2]}")

test_df['pet_temperature'] = [get_temp(item['comments']) for item in df]

## Adding comments

In [163]:
# Add comments, remove any "Ref. some number" from the original comment
def clean_comments(comment):
    if comment is None:
        return None
    # Remove "Ref. some number" using regex
    cleaned_comment = re.sub(r'\(Ref\.\s*\d+(?:,\s*\d+)*\)', '', comment)
    return cleaned_comment.strip()

test_df['pet_comments'] = [clean_comments(item['comments']) for item in df]

## Adding missing fish

In [170]:
with open("failed_species_names.json", encoding="utf-8") as f:
    missing_json = json.load(f)

pprint.pprint(missing_json[0])

{'id': 1554, 'name': 'Laeviscutella dekimpei'}


## Final file output

In [175]:
# Convert to df
missing_df = pd.DataFrame(missing_json)
missing_df = missing_df.rename(columns={'id': 'pet_id', 'name': 'pet_scientific_name'})
missing_df['pet_id'] = missing_df['pet_id'].astype(str) + 'f'

# Add contents of missing_df to test_df, adding 'f' to the end to indicate fishbase id
test_df = pd.concat([test_df, missing_df], ignore_index=True)

In [180]:
pprint.pprint(test_df[test_df['pet_id'] == '1554f'])

      pet_id     pet_scientific_name pet_vernacular_name pet_genus pet_family  \
17432  1554f  Laeviscutella dekimpei                 NaN       NaN        NaN   

      pet_body_shape pet_traits  pet_max_length  pet_max_weight  \
17432            NaN        NaN             NaN             NaN   

       pet_longevity pet_temperature pet_migration_type pet_danger  \
17432            NaN             NaN                NaN        NaN   

      pet_is_native pet_comments  
17432           NaN          NaN  


In [176]:
###TEST CELL
test_df.head(10)

,pet_id,pet_scientific_name,pet_vernacular_name,pet_genus,pet_family,pet_body_shape,pet_traits,pet_max_length,pet_max_weight,pet_longevity,pet_temperature,pet_migration_type,pet_danger,pet_is_native,pet_comments
0,2f,Oreochromis niloticus,Nile Tilapia,Oreochromis,Cichlidae,Fusiform / Normal,NaN,60.0,4.324,12.0,13.5-33 °C,Potamodromous,Harmless; No Special Ability,Not Native,Adults occur in a wide variety of freshwater h...
1,3f,Oreochromis mossambicus,Mozambique Tilapia,Oreochromis,Cichlidae,Fusiform / Normal,NaN,39.0,1.130,11.0,17-35 °C,Amphidromous,Harmless; No Special Ability,Not Native,Adults thrive in standing waters . Inhabits re...
2,35f,Anguilla anguilla,European Eel,Anguilla,Anguillidae,Eel-Like,NaN,133.0,6.599,55.0,20-25 °C,Catadromous,Harmless; No Special Ability,Not Native,Inhabits all types of benthic habitats from st...
3,63f,Dicentrarchus labrax,European Seabass,Dicentrarchus,Moronidae,Fusiform / Normal,NaN,103.0,12.000,30.0,None,Catadromous,Harmless; No Special Ability,Not Native,"Adults manifest demersal behavior, inhabit coa..."
4,73f,Gobius paganellus,Rock Goby,Gobius,Gobiidae,Elongated,NaN,13.0,NaN,10.0,None,Amphidromous,Harmless; No Special Ability,Not Native,"Predominantly marine, but may enter freshwater..."
5,79f,Ctenopharyngodon idella,Grass Carp,Ctenopharyngodon,Xenocyprididae,Fusiform / Normal,NaN,150.0,45.000,21.0,None,Potamodromous,Harmless; No Special Ability,Not Native,"Adults occur in lakes, ponds, pools and backwa..."
6,80f,Chanos chanos,Milkfish,Chanos,Chanidae,Elongated,NaN,124.0,14.000,15.0,None,Amphidromous,Harmless; No Special Ability,Native,Adults are found in offshore marine waters and...
7,82f,Labeo rohita,Roho Labeo,Labeo,Cyprinidae,Fusiform / Normal,NaN,200.0,45.000,10.0,None,Potamodromous,Harmless; No Special Ability,Not Native,Adults inhabit rivers . A diurnal species and...
8,101f,Alosa alosa,Allis Shad,Alosa,Alosidae,Fusiform / Normal,NaN,83.0,4.000,10.0,None,Anadromous,Harmless; No Special Ability,Not Native,"Amphihaline species, schooling and strongly mi..."
9,105f,Alosa immaculata,Pontic Shad,Alosa,Alosidae,Fusiform / Normal,NaN,39.0,NaN,7.0,6-9 °C,Anadromous,Harmless; No Special Ability,Not Native,"Thai species is pelagic at sea, in deep water...."
